# Notebook 81: AMBIGUOUS-Treatment Benchmark after Notebook 79

Purpose: quantify the visual observation from Notebook 79 that disabling `AMBIGUOUS -> Telea` appears to retain materially more printed notation, at the cost of retaining more annotation.

This notebook **does not alter routing, masks, p_print, p_remove, SAFE_REMOVE, or Ring-2 geometry**. It compares five treatment arms only at pixels already routed `AMBIGUOUS` by the frozen V7.8/V8.x route construction.

## Arms

- **A0 — TELEA (incumbent treatment):** `AMBIGUOUS -> Telea`.
- **A1 — OBSERVED:** `AMBIGUOUS -> exact observed byte`.
- **A2 — PROBABILITY BLEND:** `AMBIGUOUS -> p_print * observed + (1-p_print) * Telea`. No new threshold is introduced.
- **A3 — PIXEL STRUCTURAL GATE:** preserve an AMBIGUOUS pixel when calibrated `p_print >= 0.50`; otherwise use Telea. `0.50` is fixed before truth is loaded and is diagnostic only.
- **A4 — COMPONENT GATE:** 8-connected AMBIGUOUS components are preserved whole when the component median calibrated `p_print >= 0.50`; otherwise the entire component uses Telea.

For all arms:

- `REMOVE -> frozen local paper estimate`
- `PRESERVE -> observed`
- V8.9 Ring-2 exact observed-byte preservation is applied last.
- Pixels outside the frozen dilation-2 authority remain observed.

The notebook first locks all five observable outputs. Clean truth is loaded only afterwards for the paired development cohort. HITL pages receive no truth-derived scoring and are used for whole-page visual diagnostics only.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn joblib torch torchvision pyarrow matplotlib


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, importlib.util, sys, gc
import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V89_OPERATOR_PATH=ROOT/'v8_9_release'/'v8_9_operator.py'
P_PRINT_BUNDLE=ROOT/'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE=ROOT/'m5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'
NB72_ROOT=ROOT/'notebook72_frozen_v8_8_annotated_only_operational_trial'
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
CLEAN_DIR=ROOT/'source'/'clean'
ALIGNED_DIR=ROOT/'aligned'
SEED_DIRS=[ROOT/'masks'/'corrected']
MANIFEST_PATH=ROOT/'development_score_manifest.csv'

RUN=ROOT/'notebook81_ambiguous_treatment_benchmark'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook81')
for p in [RUN,RUN/'pages',RUN/'tables',RUN/'visual',RUN/'audit',RUN/'upload']:
    p.mkdir(parents=True,exist_ok=False)

ARMS=['A0_TELEA','A1_OBSERVED','A2_PPRINT_BLEND','A3_PIXEL_GATE','A4_COMPONENT_GATE']
PPRINT_GATE=0.50
EXPECTED_HITL=151


def require(x,m):
    if not bool(x): raise RuntimeError(m)

def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()

def latest(root,required):
    hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in required)]
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def image_map(folder):
    if not Path(folder).exists(): return {}
    return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}

def read_gray(p):
    im=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    require(im is not None,f'Unreadable image: {p}')
    return im

def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

require(V89_OPERATOR_PATH.exists(),f'Missing {V89_OPERATOR_PATH}')
spec=importlib.util.spec_from_file_location('v8_9_operator',V89_OPERATOR_PATH)
mod=importlib.util.module_from_spec(spec);sys.modules['v8_9_operator']=mod;spec.loader.exec_module(mod)
op=mod.load_v89_operator(P_PRINT_BUNDLE,REMOVE_BUNDLE)
require(type(op).__name__=='V89Operator',f'Expected V89Operator, got {type(op).__name__}')
print('Operator:',type(op).__name__)
print('Run:',RUN)


## Build both cohorts

HITL uses the reviewed Notebook-72 roster. Paired development uses the canonical aligned annotated page, clean page, reviewed seed and V7.5 cache. Clean pages are *registered now but are not opened* until after the observable prediction lock closes.


In [ ]:
NB72=latest(NB72_ROOT,['inputs/runtime_roster.csv'])
require(NB72 is not None,'Complete Notebook72 runtime roster not found.')
hitl=pd.read_csv(NB72/'inputs/runtime_roster.csv')
require(len(hitl)==EXPECTED_HITL,f'Expected 151 HITL pages, found {len(hitl)}')
hitl_rows=[]
for _,r in hitl.iterrows():
    hitl_rows.append({'page_key':'hitl__annotated_only_'+str(r.page),'page':str(r.page),'cohort':'HITL_ANNOTATED_ONLY','source_group':str(r.get('cohort','HITL')),'observed_path':str(r.observed_path),'seed_path':str(r.seed_mask_path),'clean_path':None})

clean_map=image_map(CLEAN_DIR);obs_map=image_map(ALIGNED_DIR)
seed_map={}
for folder in SEED_DIRS:
    seed_map.update(image_map(folder))
manifest=pd.read_csv(MANIFEST_PATH,dtype=str).fillna('')
meta=manifest.set_index('page')
paired_pages=sorted(set(clean_map)&set(obs_map)&set(seed_map)&set(meta.index.astype(str)))
paired_rows=[{'page_key':'paired__'+p,'page':p,'cohort':'PAIRED_DEVELOPMENT','source_group':str(meta.loc[p,'source_group']),'observed_path':str(obs_map[p]),'seed_path':str(seed_map[p]),'clean_path':str(clean_map[p])} for p in paired_pages]
roster=pd.DataFrame(hitl_rows+paired_rows)
roster.to_csv(RUN/'tables'/'roster.csv',index=False)
print(roster.groupby('cohort').size())


## Arm constructor

This is the key experimental isolation. Frozen route masks, frozen paper surface, frozen Telea result and frozen calibrated `p_print` are computed once. Only the treatment of already-AMBIGUOUS pixels changes. V8.9 Ring-2 preservation is then applied identically to all arms.


In [ ]:
def build_arms(observed,seed,page):
    # Run frozen V8.9 once to obtain p_print and exact Ring-2 geometry.
    v89=op.run_page(observed,seed,page=page)

    mask=mod.prepare_mask(seed,2)
    telea=mod.telea_reconstruct(observed,mask)
    baseline,routes=mod.geometry_t80(observed,mask,telea)
    preserve=np.asarray(routes['preserve'],bool)
    remove=np.asarray(routes['remove'],bool)
    ambiguous=np.asarray(routes['ambiguous'],bool)
    paper=np.asarray(routes['paper'])
    ring2=np.asarray(v89.ring2_mask,bool)
    p_print=np.asarray(v89.p_print,dtype=np.float32)

    # If p_print is unavailable at an AMBIGUOUS coordinate, probability-based
    # arms fail back to incumbent Telea rather than guessing preservation.
    pp=np.where(np.isfinite(p_print),np.clip(p_print,0.0,1.0),0.0)

    base=observed.copy()
    base[remove]=paper[remove]
    base[preserve]=observed[preserve]

    arms={}
    a0=base.copy();a0[ambiguous]=telea[ambiguous]
    a1=base.copy();a1[ambiguous]=observed[ambiguous]

    a2=base.copy()
    blend=np.rint(pp.astype(np.float32)*observed.astype(np.float32)+(1.0-pp.astype(np.float32))*telea.astype(np.float32)).clip(0,255).astype(np.uint8)
    a2[ambiguous]=blend[ambiguous]

    gate=ambiguous & (pp>=PPRINT_GATE)
    a3=base.copy();a3[ambiguous]=telea[ambiguous];a3[gate]=observed[gate]

    a4=base.copy();a4[ambiguous]=telea[ambiguous]
    n,labels=cv2.connectedComponents(ambiguous.astype(np.uint8),connectivity=8)
    component_preserve=np.zeros_like(ambiguous,bool)
    for cid in range(1,n):
        comp=labels==cid
        vals=pp[comp]
        if len(vals) and float(np.median(vals))>=PPRINT_GATE:
            component_preserve[comp]=True
    a4[component_preserve]=observed[component_preserve]

    arms['A0_TELEA']=a0;arms['A1_OBSERVED']=a1;arms['A2_PPRINT_BLEND']=a2;arms['A3_PIXEL_GATE']=a3;arms['A4_COMPONENT_GATE']=a4

    # Apply identical V8.9 Ring-2 exact preservation last.
    for k in arms:
        arms[k][ring2]=observed[ring2]
        require(np.array_equal(arms[k][mask==0],observed[mask==0]),f'{page} {k}: outside-mask drift')
        require(np.array_equal(arms[k][ring2],observed[ring2]),f'{page} {k}: Ring2 mismatch')

    return {
        'arms':arms,'mask':mask>0,'preserve':preserve,'remove':remove,'ambiguous':ambiguous,
        'paper':paper,'telea':telea,'p_print':p_print,'ring2':ring2,'pixel_gate':gate,'component_gate':component_preserve
    }


## Generate and lock all observable outputs

No clean page is opened in this cell. The outputs, route masks and hashes are frozen before truth scoring.


In [ ]:
# ============================================================
# Generate and lock all observable outputs
#
# Zero-authority pages are valid zero-exposure controls.
# They are recorded explicitly and skipped before build_arms().
# No clean truth is read in this cell.
# ============================================================

output_rows = []


for _, r in roster.iterrows():

    key = str(r.page_key)
    page = str(r.page)
    cohort = str(r.cohort)
    source_group = str(r.source_group)

    obs = read_gray(r.observed_path)
    seed = read_gray(r.seed_path)

    require(
        obs.shape == seed.shape,
        f"{key}: observed/seed mismatch",
    )


    # --------------------------------------------------------
    # Zero-authority / hard-negative handling
    #
    # If the reviewed seed is empty, there is no authorised
    # intervention population and therefore no M3/AMBIGUOUS
    # treatment experiment to run.
    # --------------------------------------------------------

    seed_bool = np.asarray(seed) > 0
    seed_pixels = int(seed_bool.sum())

    if seed_pixels == 0:

        print(
            f"{key}: SKIP - zero-authority page "
            "(reviewed seed contains 0 pixels)"
        )

        output_rows.append({
            "page_key": key,
            "page": page,
            "cohort": cohort,
            "source_group": source_group,

            "status":
                "SKIPPED_ZERO_AUTHORITY",

            "skip_reason":
                "reviewed_seed_has_zero_pixels",

            "seed_pixels":
                0,

            "ambiguous_pixels":
                0,

            "ring2_pixels":
                0,

            "pixel_gate_pixels":
                0,

            "component_gate_pixels":
                0,

            **{
                f"{arm}_changed_vs_observed": 0
                for arm in ARMS
            },
        })

        continue


    # --------------------------------------------------------
    # Build A0-A4 on an actually exposed page.
    # --------------------------------------------------------

    try:

        z = build_arms(
            obs,
            seed,
            page,
        )

    except RuntimeError as e:

        # ----------------------------------------------------
        # A non-empty seed can still theoretically lead to no
        # effective M3 / AMBIGUOUS population.
        #
        # Only convert the known empty-population conditions
        # into an explicit skip. All other RuntimeErrors still
        # fail normally.
        # ----------------------------------------------------

        msg = str(e).lower()

        known_empty_population = (
            "no m3" in msg
            or "m3 pixels" in msg
            or "no ambiguous" in msg
            or "ambiguous pixels" in msg
        )

        if not known_empty_population:
            raise

        print(
            f"{key}: SKIP - no experimental "
            f"M3/AMBIGUOUS population ({e})"
        )

        output_rows.append({
            "page_key": key,
            "page": page,
            "cohort": cohort,
            "source_group": source_group,

            "status":
                "SKIPPED_NO_AMBIGUOUS_POPULATION",

            "skip_reason":
                str(e),

            "seed_pixels":
                seed_pixels,

            "ambiguous_pixels":
                0,

            "ring2_pixels":
                0,

            "pixel_gate_pixels":
                0,

            "component_gate_pixels":
                0,

            **{
                f"{arm}_changed_vs_observed": 0
                for arm in ARMS
            },
        })

        gc.collect()
        continue


    # --------------------------------------------------------
    # Defensive population check.
    #
    # Even if build_arms() returns normally, an empty
    # AMBIGUOUS population contributes no evidence to this
    # treatment benchmark.
    # --------------------------------------------------------

    ambiguous_pixels = int(
        np.asarray(
            z["ambiguous"],
            dtype=bool,
        ).sum()
    )

    if ambiguous_pixels == 0:

        print(
            f"{key}: SKIP - "
            "AMBIGUOUS population contains 0 pixels"
        )

        output_rows.append({
            "page_key": key,
            "page": page,
            "cohort": cohort,
            "source_group": source_group,

            "status":
                "SKIPPED_NO_AMBIGUOUS_POPULATION",

            "skip_reason":
                "ambiguous_population_zero",

            "seed_pixels":
                seed_pixels,

            "ambiguous_pixels":
                0,

            "ring2_pixels":
                int(
                    np.asarray(
                        z["ring2"],
                        dtype=bool,
                    ).sum()
                ),

            "pixel_gate_pixels":
                0,

            "component_gate_pixels":
                0,

            **{
                f"{arm}_changed_vs_observed": 0
                for arm in ARMS
            },
        })

        del z
        gc.collect()
        continue


    # --------------------------------------------------------
    # Write observable outputs only for genuinely exposed
    # experimental pages.
    # --------------------------------------------------------

    d = RUN / "pages" / key

    d.mkdir(
        parents=True,
        exist_ok=False,
    )

    cv2.imwrite(
        str(d / "observed.png"),
        obs,
    )

    cv2.imwrite(
        str(d / "seed.png"),
        seed,
    )

    cv2.imwrite(
        str(d / "ambiguous.png"),
        (
            z["ambiguous"].astype(np.uint8)
            * 255
        ),
    )

    cv2.imwrite(
        str(d / "ring2.png"),
        (
            z["ring2"].astype(np.uint8)
            * 255
        ),
    )


    # --------------------------------------------------------
    # Diagnostic arrays remain truth-free.
    # --------------------------------------------------------

    np.savez_compressed(
        d / "diagnostic_arrays.npz",

        p_print=
            z["p_print"],

        pixel_gate=
            z["pixel_gate"],

        component_gate=
            z["component_gate"],
    )


    # --------------------------------------------------------
    # Write all A0-A4 arm outputs.
    # --------------------------------------------------------

    for arm, img in z["arms"].items():

        cv2.imwrite(
            str(d / f"{arm}.png"),
            img,
        )


    # --------------------------------------------------------
    # Record completed experimental page.
    # --------------------------------------------------------

    output_rows.append({
        "page_key":
            key,

        "page":
            page,

        "cohort":
            cohort,

        "source_group":
            source_group,

        "status":
            "PROCESSED",

        "skip_reason":
            "",

        "seed_pixels":
            seed_pixels,

        "ambiguous_pixels":
            ambiguous_pixels,

        "ring2_pixels":
            int(
                np.asarray(
                    z["ring2"],
                    dtype=bool,
                ).sum()
            ),

        "pixel_gate_pixels":
            int(
                np.asarray(
                    z["pixel_gate"],
                    dtype=bool,
                ).sum()
            ),

        "component_gate_pixels":
            int(
                np.asarray(
                    z["component_gate"],
                    dtype=bool,
                ).sum()
            ),

        **{
            f"{arm}_changed_vs_observed":
                int(
                    np.sum(
                        img != obs
                    )
                )

            for arm, img
            in z["arms"].items()
        },
    })


    print(
        f"{key}: PROCESSED - "
        f"seed={seed_pixels:,}, "
        f"ambiguous={ambiguous_pixels:,}"
    )

    del z
    gc.collect()


# ============================================================
# Observable-output summary
# ============================================================

outputs = pd.DataFrame(
    output_rows
)

outputs.to_csv(
    RUN
    / "tables"
    / "observable_output_summary.csv",

    index=False,
)


# ------------------------------------------------------------
# Explicit processing / exclusion report
# ------------------------------------------------------------

processed = outputs[
    outputs["status"] == "PROCESSED"
].copy()

skipped = outputs[
    outputs["status"] != "PROCESSED"
].copy()


skip_summary = (
    skipped
    .groupby(
        [
            "cohort",
            "status",
            "skip_reason",
        ],
        dropna=False,
    )
    .size()
    .reset_index(
        name="pages"
    )
)

skip_summary.to_csv(
    RUN
    / "tables"
    / "observable_output_exclusions.csv",

    index=False,
)


print()
print("=" * 72)
print("OBSERVABLE OUTPUT GENERATION COMPLETE")
print("=" * 72)

print(
    "Roster pages:",
    len(outputs),
)

print(
    "Processed pages:",
    len(processed),
)

print(
    "Skipped zero-exposure pages:",
    len(skipped),
)

print()

if len(skip_summary):
    print(
        skip_summary.to_string(
            index=False
        )
    )


# ============================================================
# Hash all files that actually exist
# ============================================================

hash_rows = []

for p in sorted(
    (RUN / "pages").rglob("*")
):

    if not p.is_file():
        continue

    hash_rows.append({
        "path":
            str(
                p.relative_to(RUN)
            ),

        "bytes":
            int(
                p.stat().st_size
            ),

        "sha256":
            sha256_file(p),
    })


hashes = pd.DataFrame(
    hash_rows
)

hashes.to_csv(
    RUN
    / "audit"
    / "prediction_hashes.csv",

    index=False,
)


# ============================================================
# Prediction lock
#
# Important distinction:
#
# roster_pages
#     = every declared input page
#
# processed_pages
#     = pages that actually contain an AMBIGUOUS treatment
#       population
#
# skipped_pages
#     = zero-authority / zero-exposure controls
# ============================================================

lock = {
    "state":
        "CLOSED_BEFORE_CLEAN_TRUTH",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "roster_pages":
        int(
            len(outputs)
        ),

    "processed_pages":
        int(
            len(processed)
        ),

    "skipped_pages":
        int(
            len(skipped)
        ),

    "processed_pages_by_cohort":
        {
            str(k): int(v)
            for k, v in
            processed[
                "cohort"
            ].value_counts().items()
        },

    "skipped_pages_by_cohort":
        {
            str(k): int(v)
            for k, v in
            skipped[
                "cohort"
            ].value_counts().items()
        },

    "skip_reasons":
        (
            skip_summary
            .to_dict(
                orient="records"
            )
            if len(skip_summary)
            else []
        ),

    "arms":
        ARMS,

    "pprint_gate":
        PPRINT_GATE,

    "hash_table_sha256":
        hashlib.sha256(
            hashes.to_csv(
                index=False
            ).encode()
        ).hexdigest(),

    "observable_output_summary_sha256":
        sha256_file(
            RUN
            / "tables"
            / "observable_output_summary.csv"
        ),

    "exclusions_sha256":
        sha256_file(
            RUN
            / "tables"
            / "observable_output_exclusions.csv"
        ),

    "operator_sha256":
        sha256_file(
            V89_OPERATOR_PATH
        ),

    "clean_truth_read":
        False,
}


(
    RUN / "prediction_lock.json"
).write_text(
    json.dumps(
        lock,
        indent=2,
    )
)


print()
print(
    json.dumps(
        lock,
        indent=2,
    )
)

## Post-lock paired truth scoring

Only now are clean pages opened. Primary quantities are calculated **inside AMBIGUOUS coordinates only**, because that is the sole experimental treatment population. A one-pixel tolerant clean-print map is used for genuine-print support. Dark observed pixels with no clean-print support are scored as annotation-like; remaining pixels are truth-uncertain/other and reported separately.


In [ ]:
require(json.loads((RUN/'prediction_lock.json').read_text())['state']=='CLOSED_BEFORE_CLEAN_TRUTH','Prediction lock not closed.')
score_rows=[];pixel_parts=[]
paired=roster[roster.cohort=='PAIRED_DEVELOPMENT'].copy()
for _,r in paired.iterrows():
    key=str(r.page_key);page=str(r.page);d=RUN/'pages'/key
    obs=read_gray(r.observed_path);clean=read_gray(r.clean_path);seed=read_gray(r.seed_path)
    clean_print=printed_ink(clean)
    clean_tol=cv2.dilate(clean_print.astype(np.uint8),np.ones((3,3),np.uint8))>0
    z=build_arms(obs,seed,page)
    amb=z['ambiguous']
    annotation=amb & (obs<200) & ~clean_tol
    print_truth=amb & clean_tol
    uncertain=amb & ~(annotation|print_truth)

    for arm,img in z['arms'].items():
        changed_from_observed=amb & (img!=obs)
        lost_print=print_truth & changed_from_observed
        removed_annotation=annotation & changed_from_observed
        retained_print=print_truth & ~changed_from_observed
        retained_annotation=annotation & ~changed_from_observed
        score_rows.append({'page_key':key,'page':page,'source_group':r.source_group,'arm':arm,'ambiguous_pixels':int(amb.sum()),'truth_print_pixels':int(print_truth.sum()),'truth_annotation_pixels':int(annotation.sum()),'truth_uncertain_pixels':int(uncertain.sum()),'print_changed_from_observed':int(lost_print.sum()),'print_preserved_observed':int(retained_print.sum()),'annotation_changed_from_observed':int(removed_annotation.sum()),'annotation_retained_observed':int(retained_annotation.sum())})

    ys,xs=np.where(amb)
    if len(ys):
        part=pd.DataFrame({'page_key':key,'page':page,'source_group':r.source_group,'y':ys,'x':xs,'truth_print':print_truth[ys,xs],'truth_annotation':annotation[ys,xs],'truth_uncertain':uncertain[ys,xs],'p_print':z['p_print'][ys,xs]})
        for arm,img in z['arms'].items(): part[arm+'_equals_observed']=img[ys,xs]==obs[ys,xs]
        pixel_parts.append(part)
    del z;gc.collect()

scores=pd.DataFrame(score_rows);scores.to_csv(RUN/'tables'/'paired_truth_by_page.csv',index=False)
pixels=pd.concat(pixel_parts,ignore_index=True) if pixel_parts else pd.DataFrame();pixels.to_parquet(RUN/'tables'/'paired_ambiguous_pixel_truth.parquet',index=False)
pooled=scores.groupby('arm',as_index=False).agg(pages=('page_key','nunique'),ambiguous_pixels=('ambiguous_pixels','sum'),truth_print_pixels=('truth_print_pixels','sum'),truth_annotation_pixels=('truth_annotation_pixels','sum'),truth_uncertain_pixels=('truth_uncertain_pixels','sum'),print_changed_from_observed=('print_changed_from_observed','sum'),print_preserved_observed=('print_preserved_observed','sum'),annotation_changed_from_observed=('annotation_changed_from_observed','sum'),annotation_retained_observed=('annotation_retained_observed','sum'))
pooled['print_preservation_rate']=pooled.print_preserved_observed/pooled.truth_print_pixels.clip(lower=1)
pooled['annotation_removal_rate']=pooled.annotation_changed_from_observed/pooled.truth_annotation_pixels.clip(lower=1)
pooled.to_csv(RUN/'tables'/'pooled_arm_summary.csv',index=False)
display(pooled)


## Visual summaries

The first chart deliberately treats print preservation and annotation removal as separate axes. There is no single utility score. The second chart shows page-level trade-offs. Contact sheets use selected paired pages and HITL pages with the largest A0↔A1 visual difference.


In [ ]:
fig,ax=plt.subplots(figsize=(8,6))
for _,r in pooled.iterrows():
    ax.scatter(r.annotation_removal_rate,r.print_preservation_rate,s=80,label=r.arm)
    ax.text(r.annotation_removal_rate,r.print_preservation_rate,r.arm.replace('_',' '),fontsize=8)
ax.set_xlabel('Annotation removal rate within AMBIGUOUS truth-annotation pixels')
ax.set_ylabel('Genuine print preservation rate within AMBIGUOUS truth-print pixels')
ax.set_xlim(0,1.02);ax.set_ylim(0,1.02);ax.grid(True,alpha=.25);plt.tight_layout();plt.savefig(RUN/'visual'/'arm_tradeoff.png',dpi=160);plt.close(fig)

trade=scores.pivot(index='page_key',columns='arm',values=['print_preserved_observed','annotation_retained_observed']).reset_index()
# Flatten for easy downstream reading.
trade.columns=['__'.join([str(x) for x in c if str(x)]) if isinstance(c,tuple) else str(c) for c in trade.columns]
trade.to_csv(RUN/'tables'/'page_tradeoff_wide.csv',index=False)

obs_summary=pd.read_csv(RUN/'tables'/'observable_output_summary.csv')
obs_summary['A0_A1_delta']=0
for i,r in obs_summary.iterrows():
    d=RUN/'pages'/str(r.page_key);a0=read_gray(d/'A0_TELEA.png');a1=read_gray(d/'A1_OBSERVED.png');obs_summary.loc[i,'A0_A1_delta']=int(np.sum(a0!=a1))
obs_summary=obs_summary.sort_values('A0_A1_delta',ascending=False)
obs_summary.to_csv(RUN/'tables'/'pages_ranked_by_A0_A1_difference.csv',index=False)

def make_contacts(frame,name,n=24,per=6):
    qs=frame.head(n)
    paths=[]
    for start in range(0,len(qs),per):
        q=qs.iloc[start:start+per];fig,axes=plt.subplots(len(q),6,figsize=(24,max(5,4.7*len(q))),squeeze=False)
        for rr,(_,row) in enumerate(q.iterrows()):
            d=RUN/'pages'/str(row.page_key)
            ips=[d/'observed.png']+[d/(a+'.png') for a in ARMS]
            titles=['Observed']+ARMS
            for cc,(p,t) in enumerate(zip(ips,titles)):
                im=read_gray(p);axes[rr,cc].imshow(im,cmap='gray',vmin=0,vmax=255);axes[rr,cc].set_title((str(row.page)+'\n'+t) if cc==0 else t,fontsize=7);axes[rr,cc].axis('off')
        plt.tight_layout();p=RUN/'visual'/f'{name}_{start//per+1:02d}.png';plt.savefig(p,dpi=105,bbox_inches='tight');plt.close(fig);paths.append(p)
    return paths
paired_rank=obs_summary[obs_summary.cohort=='PAIRED_DEVELOPMENT'];hitl_rank=obs_summary[obs_summary.cohort=='HITL_ANNOTATED_ONLY']
contacts=make_contacts(paired_rank,'paired_contact')+make_contacts(hitl_rank,'hitl_contact')
print('Contact sheets:',len(contacts))


## Final diagnostic verdict and upload bundle

This notebook does **not** promote an arm. It reports the Pareto-style trade-off. A later decision can favour retained musical information over cosmetic cleanliness, consistent with the visual-restorer objective.


In [ ]:
pooled=pd.read_csv(RUN/'tables'/'pooled_arm_summary.csv')
verdict={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'notebook':'81_AMBIGUOUS_Treatment_Benchmark',
    'status':'DIAGNOSTIC_COMPLETE_NO_AUTOMATIC_WINNER',
    'arms':ARMS,
    'paired_pages':int(roster[roster.cohort=='PAIRED_DEVELOPMENT'].shape[0]),
    'hitl_pages':int(roster[roster.cohort=='HITL_ANNOTATED_ONLY'].shape[0]),
    'prediction_lock':'CLOSED_BEFORE_CLEAN_TRUTH',
    'uses_clean_truth_for_inference':False,
    'promotion_authority':False,
    'interpretation':'Compare print preservation and annotation removal separately; do not collapse them into one utility score.'
}
(RUN/'notebook81_verdict.json').write_text(json.dumps(verdict,indent=2))
summary=[
    'NOTEBOOK 81 - AMBIGUOUS TREATMENT BENCHMARK',
    '',
    f'HITL pages: {verdict["hitl_pages"]}',
    f'Paired pages: {verdict["paired_pages"]}',
    '',
    'Primary output: pooled_arm_summary.csv',
    'Visual output: arm_tradeoff.png plus paired/HITL contact sheets',
    '',
    'No production promotion is authorised.'
]
summary_text='\n'.join(summary)
(RUN/'executive_summary.txt').write_text(summary_text)
for pth in [
    RUN/'executive_summary.txt',
    RUN/'notebook81_verdict.json',
    RUN/'prediction_lock.json',
    RUN/'tables'/'pooled_arm_summary.csv',
    RUN/'tables'/'paired_truth_by_page.csv',
    RUN/'tables'/'pages_ranked_by_A0_A1_difference.csv',
    RUN/'visual'/'arm_tradeoff.png',
    *contacts,
]:
    shutil.copy2(pth,RUN/'upload'/pth.name)
shutil.make_archive(str(RUN/'notebook81_ambiguous_treatment_bundle'),'zip',RUN/'upload')
print(summary_text)
display(pooled)
print('COMPLETE:',RUN)
